In [ ]:
from pydantic import BaseModel
from openai import OpenAI
from dotenv import load_dotenv

'''
OpenAI supports Pydantic models natively via the .parse() method. 
It automatically parses and validates the response into your requested object.
'''


# Load OPENAI_API_KEY from the environment — never hard-code keys
load_dotenv() #loads the content of a file named .env and setsup session env variables.

class UserInfo(BaseModel):
    name: str
    age: int
    skills: list[str]

client = OpenAI()

# Returns a parsed UserInfo instance inside the response object
completion = client.beta.chat.completions.parse(
    model="gpt-4o-2024-08-06",
    messages=[
        {"role": "user", "content": "Extract profile: Alex is a 28 year old proficient in Python and SQL."}
    ],
    response_format=UserInfo,
)

# Access directly as a Pydantic object
user: UserInfo = completion.choices[0].message.parsed
print(user.name)    # "Alex"
print(user.skills)  # ["Python", "SQL"]

Alex
['Python', 'SQL']


In [5]:
'''
Multi-Provider Solution: The instructor Library
If you work across multiple providers (OpenAI, Anthropic, Gemini, Ollama), 
the instructor Python package wraps the native SDKs and handles Pydantic conversion directly.
'''
import instructor
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

# Load OPENAI_API_KEY from the environment — never hard-code keys
load_dotenv() #loads the content of a file named .env and setsup session env variables.

class UserInfo(BaseModel):
    name: str
    age: int
    skills: list[str]

# Patch the client
client = instructor.from_openai(OpenAI())

# Directly returns the instantiated Pydantic class
user: UserInfo = client.chat.completions.create(
    model="gpt-4o",
    response_model=UserInfo,
    messages=[
        {"role": "user", "content": "Extract profile: Alex is a 28 year old proficient in Python and SQL."}
    ],
)

print(user.name)    # "Alex"
print(user.skills)  # ["Python", "SQL"]

Alex
['Python', 'SQL']


In [6]:
print(user.name)

Alex


Use this system prompt as an example to understand how we can use the pydantic class object and directly assign the details as class attributes.
"Answer the the user question in the format of content, conficdence and sources"
GEMINI EXPLAINER
Confidence and sources are not hidden in the API metadata—OpenAI API metadata strictly contains operational metrics like token usage, model version, and completion IDs.

When a prompt asks for content, confidence, and sources, the LLM attempts to output those details within the generated message body, not the API wrapper.

You can extract these fields directly as attributes of a custom Pydantic class using OpenAI's Structured Outputs.

Defining Custom Attributes via Pydantic

By defining content, confidence, and sources as fields in a Pydantic model, OpenAI will validate the model's output directly into those attributes:

**WARNING** the below code uses client.beta.chat.completions.parse which is a beta feature.


In [ ]:
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv

# Load OPENAI_API_KEY from the environment — never hard-code keys
load_dotenv() #loads the content of a file named .env and setsup session env variables.
# 1. Define the custom response schema
class AnalysisResponse(BaseModel):
    content: str = Field(description="The core answer to the query.")
    confidence: float = Field(description="Self-assessed confidence score between 0.0 and 1.0.")
    sources: list[str] = Field(description="List of cited sources, documents, or data references.")

client = OpenAI()

# 2. Call the API using structured parsing
completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Answer the the user question in the format of content, conficdence and sources."},
        {"role": "user", "content": "How is the growth rate of India?"}
    ],
    response_format=AnalysisResponse,
)

# 3. Access attributes directly on the returned object
result: AnalysisResponse = completion.choices[0].message.parsed

print(result.content)     # "India's GDP growth rate is projected..."
print(result.confidence)  # 0.92
print(result.sources)     # ["IMF World Economic Outlook", "Reserve Bank of India"]

We will try the same but work with instructor module so that our code is not locked with OpenAPI models

In [11]:
import instructor
from pydantic import BaseModel
from dotenv import load_dotenv

# Load OPENAI_API_KEY from the environment — never hard-code keys
load_dotenv() #loads the content of a file named .env and setsup session env variables.
# 1. Define the custom response schema

class AnalysisResponse(BaseModel):
    content: str
    confidence: float
    sources: list[str]

# Switch client wrapper seamlessly:

# OpenAI
from openai import OpenAI
client = instructor.from_openai(OpenAI())

# Anthropic (Claude)
# from anthropic import Anthropic
# client = instructor.from_anthropic(Anthropic())

# Google Gemini
# from google import genai
# client = instructor.from_gemini(genai.Client())

# Ollama (Local LLMs)
# from ollama import Client
# client = instructor.from_ollama(Client())

# Call remains identical across providers:
response: AnalysisResponse = client.chat.completions.create(
    model="gpt-4o",  # or "claude-3-5-sonnet-20240620", "gemini-1.5-pro", etc.
    response_model=AnalysisResponse,
    messages=[{"role": "user", "content": "How is the growth rate of India?"}],
)
print(response.content, response.confidence, response.sources)

India's economy has demonstrated robust growth in recent years. As of the last few quarters leading up to 2023, India has maintained one of the fastest growth rates among major economies. Specific numbers can vary by quarter or year, but India’s GDP growth rate has often been in the range of 6-8% annually in the pre-2023 period, according to recent government and international financial reports. This growth is driven by domestic consumption, digital economy expansion, and infrastructure investments. Additionally, various policy reforms aimed at improving the business environment have continued to support economic expansion. 0.95 ['IMF Reports', 'World Bank Data', 'Indian Government Economic Surveys']


Alternative Options for Portability

LangChain / LlamaIndex: Frameworks offer .with_structured_output(PydanticModel), abstracting away the underlying provider API.

Manual JSON Schema Construction: Convert your model via PydanticModel.model_json_schema(), pass it to the vendor's standard JSON mode parameter, and parse the raw string response using PydanticModel.model_validate_json().

['World Bank Economic Reports', 'IMF Forecasts', 'Ministry of Finance India']
